# Biology — Plant growth and soil nutrient analytics
## FDP Day 5 — AI-Powered Scientific Research with Google Colab

**Objective:** Explore nitrogen, phosphorus, potassium, pH and moisture associations with plant height and yield; build an exploratory yield model.

**Matching dataset:** `03_Biology_Plant_Growth_Soil_Nutrients.csv`

All bundled CSVs are **synthetic teaching datasets** with plausible ranges and units, not real experimental observations. Replace them with approved research data for actual studies. Avoid uploading identifiable patient or confidential data.

**Workflow:** Upload → inspect → analyse → visualise → interpret → export.


## 1. Setup and upload the matching CSV

In [ ]:
!pip -q install seaborn scikit-learn opencv-python-headless
import io, warnings, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from google.colab import files
from IPython.display import display
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid",context="notebook")
plt.rcParams["figure.figsize"]=(9,5)
uploaded=files.upload()
csv_name=next(f for f in uploaded if f.lower().endswith(".csv"))
df=pd.read_csv(io.BytesIO(uploaded[csv_name]))
print("Loaded",csv_name,"shape:",df.shape)
display(df.head()); display(df.describe(include="all").T)


## 2. Exploratory statistics and predictive modelling

In [ ]:
num=df.select_dtypes(include=np.number)
sns.heatmap(num.corr(),annot=True,cmap="vlag",center=0);plt.title("Soil and plant correlation matrix");plt.show()
fig,ax=plt.subplots(1,2,figsize=(12,4))
sns.scatterplot(data=df,x="nitrogen_mg_kg",y="yield_t_ha",hue="soil_pH",ax=ax[0],palette="viridis")
sns.scatterplot(data=df,x="soil_moisture_percent",y="plant_height_cm",ax=ax[1])
plt.tight_layout();plt.show()
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
features=["nitrogen_mg_kg","phosphorus_mg_kg","potassium_mg_kg","soil_pH","soil_moisture_percent"]
Xtr,Xte,ytr,yte=train_test_split(df[features],df.yield_t_ha,test_size=.25,random_state=42)
model=RandomForestRegressor(n_estimators=200,random_state=42).fit(Xtr,ytr);pred=model.predict(Xte)
print("MAE",round(mean_absolute_error(yte,pred),2),"RMSE",round(mean_squared_error(yte,pred)**.5,2),"R2",round(r2_score(yte,pred),3))
sns.barplot(x=model.feature_importances_,y=features);plt.title("Feature importance");plt.show()


## Research interpretation checklist
- State the research question, data source, sample size and units.
- Explain two findings supported by the charts.
- Identify uncertainty, confounders, measurement limitations and possible bias.
- State what additional data or validation is required before publication.
- Do not claim causation from correlation or clinical/biological diagnosis from simple image segmentation.


In [ ]:
# Export a copy of the data with any new analysis columns
df.to_csv("processed_research_data.csv",index=False)
files.download("processed_research_data.csv")
